# Week 10: Advanced Simulation with ASE and CHGNet

In this week's lab, we will combine the **Atomic Simulation Environment (ASE)** with **CHGNet** (a universal graph neural network potential for materials) to perform practical, high-level atomistic simulations.

We will cover two main applications:
1. **Adsorption Energy Calculation**: How strongly does a molecule bind to a surface?
2. **Ionic Conductivity via Molecular Dynamics (MD)**: How do we extract room-temperature conductivity from high-temperature simulations?

### Environment Setup
First, let's make sure we have the necessary libraries installed.

In [ ]:
!pip install -q ase chgnet pymatgen matplotlib mp-api scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const

from ase import Atoms
from ase.build import fcc111, molecule, add_adsorbate, bulk
from ase.optimize import FIRE
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.visualize.plot import plot_atoms
from ase import units

from chgnet.model.dynamics import CHGNetCalculator

## Part 1: Adsorption Energy Calculation

Adsorption is a critical process in catalysis, gas sensors, and energy storage. The adsorption energy ($E_{ads}$) tells us whether the process is energetically favorable.

It is calculated as:
$$E_{ads} = E_{system} - (E_{surface} + E_{adsorbate})$$

Where:
- $E_{system}$: Energy of the surface with the adsorbate attached.
- $E_{surface}$: Energy of the bare surface.
- $E_{adsorbate}$: Energy of the isolated molecule.

A negative $E_{ads}$ means the adsorption is exothermic (favorable).

### 1.1 Building the Systems

In [ ]:
# Initialize CHGNet Calculator
calculator = CHGNetCalculator()

# 1. Build the Pt(111) surface (slab)
# We create a 3x3x3 supercell of the fcc(111) surface
slab = fcc111('Pt', size=(3, 3, 3), vacuum=10.0)
slab.calc = calculator

# 2. Build the adsorbate (CO molecule)
adsorbate = molecule('CO')
# Add vacuum box around the molecule for calculation
adsorbate.set_cell([15, 15, 15])
adsorbate.center()
adsorbate.calc = calculator

In [ ]:
# Visualize the initial structures
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plot_atoms(slab, ax[0], rotation=('-75x, 45y, 10z'))
ax[0].set_title('Pt(111) Slab')
plot_atoms(adsorbate, ax[1])
ax[1].set_title('CO Molecule')
plt.show()

### 1.2 Relaxing the Isolated Systems
Before combining them, we must find the lowest energy (relaxed) states of the bare slab and the isolated molecule.

In [ ]:
print("Relaxing bare Pt(111) slab...")
opt_slab = FIRE(slab, trajectory='pt_slab.traj', logfile=None)
opt_slab.run(fmax=0.05)
e_slab = slab.get_potential_energy()
print(f"Energy of bare slab: {e_slab:.2f} eV\n")

print("Relaxing CO molecule...")
opt_mol = FIRE(adsorbate, trajectory='co_mol.traj', logfile=None)
opt_mol.run(fmax=0.05)
e_adsorbate = adsorbate.get_potential_energy()
print(f"Energy of isolated CO: {e_adsorbate:.2f} eV\n")

### 1.3 Placing Adsorbate and Final Relaxation
Now we place the relaxed CO molecule onto the Pt(111) surface and relax the combined system.

In [ ]:
# Copy the relaxed slab
system = slab.copy()

# Add the adsorbate to the 'ontop' site with a height of 1.8 Å
add_adsorbate(system, adsorbate, height=1.8, position='ontop')

# Set calculator and relax
system.calc = calculator
print("Relaxing combined Pt(111) + CO system...")
opt_sys = FIRE(system, trajectory='pt_co_system.traj', logfile=None)
opt_sys.run(fmax=0.05)
e_system = system.get_potential_energy()

print(f"Energy of combined system: {e_system:.2f} eV")

In [ ]:
# Visualize the relaxed combined system
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plot_atoms(system, ax[0], rotation=('-75x, 45y, 10z'))
ax[0].set_title('Side View')
plot_atoms(system, ax[1], rotation=('90x, 0y, 0z'))
ax[1].set_title('Top View')
plt.show()

In [ ]:
# Calculate Adsorption Energy
e_ads = e_system - (e_slab + e_adsorbate)
print(f"\nCalculated Adsorption Energy (E_ads): {e_ads:.3f} eV")

### 🏋️‍♂️ Exercise 1: Try a Different Adsorption Site
We placed the CO molecule on the 'ontop' site. What happens if we place it on the 'fcc' or 'hcp' hollow site?
1. Copy the code from section 1.3.
2. Change `position='ontop'` to `position='fcc'`.
3. Relax the new system and calculate its adsorption energy.
4. Which site is more stable?

In [ ]:
# Your code for Exercise 1 here:



## Part 2: Ionic Conductivity via Molecular Dynamics (MD)

To compute the ionic conductivity of a solid-state electrolyte at room temperature (300 K), direct Molecular Dynamics is often too slow because ions don't diffuse fast enough to gather statistics in picoseconds.

**The Workflow:**
1. Run MD at **high temperatures** (e.g., 800 K, 1000 K, 1200 K) where diffusion is fast.
2. Calculate the **Diffusion Coefficient ($D$)** for each temperature using Mean Squared Displacement (MSD).
3. Create an **Arrhenius plot** ($\ln(D)$ vs $1/T$) and extrapolate to room temperature (300 K) to find $D_{300K}$.
4. Use the **Nernst-Einstein equation** to convert $D_{300K}$ into Ionic Conductivity ($\sigma$).

### 2.1 Retrieving Structures from Materials Project
We will retrieve $\beta$-Li$_3$PS$_4$ (LPS), a famous solid-state electrolyte, using the `mp-api` package.
You need an API key from your [Materials Project Dashboard](https://materialsproject.org/dashboard).

*Note: If you don't provide an API key, it will fall back to a built-in Li bulk structure.*

In [ ]:
from mp_api.client import MPRester
from pymatgen.io.ase import AseAtomsAdaptor

# 1. Enter your Materials Project API Key here:
api_key = ""  # Leave empty to use the fallback structure

if api_key:
    try:
        # The MP-ID for beta-Li3PS4 is mp-985592
        with MPRester(api_key) as mpr:
            pmg_struct = mpr.get_structure_by_material_id("mp-985592")
            print(f"Successfully fetched: {pmg_struct.composition.reduced_formula}")
            
        solid_structure = AseAtomsAdaptor.get_atoms(pmg_struct)
        md_supercell = solid_structure * (1, 1, 2) # 64 atoms for LPS
        
    except Exception as e:
        print(f"Failed to fetch from MP: {e}")
        print("Falling back to simple Li bcc structure...")
        li_bulk = bulk('Li', 'bcc', cubic=True)
        md_supercell = li_bulk * (3, 3, 3)
else:
    print("No API key provided. Using fallback Li bcc structure...")
    li_bulk = bulk('Li', 'bcc', cubic=True)
    md_supercell = li_bulk * (3, 3, 3)

md_supercell.calc = calculator

# Identify Lithium indices for later MSD calculation
symbols = np.array(md_supercell.get_chemical_symbols())
li_indices = np.where(symbols == 'Li')[0]
print(f"Supercell has {len(symbols)} total atoms ({len(li_indices)} are Li).")

### 2.2 Running High-Temperature MD Loop
We loop over multiple high temperatures, run a short MD simulation, and extract the diffusion coefficient $D$.

In [ ]:
temperatures_K = [800, 1000, 1200]
diffusion_coeffs = []

for T in temperatures_K:
    print(f"\n--- Running MD at {T} K ---")
    
    # Set initial velocities corresponding to T
    MaxwellBoltzmannDistribution(md_supercell, T * units.kB)
    
    # Langevin thermostat
    time_step = 2.0 * units.fs
    friction = 0.02 / units.fs
    md = Langevin(md_supercell, time_step, T * units.kB, friction, logfile=None)
    
    positions = []
    times = []
    
    def collect_data():
        positions.append(md_supercell.get_positions())
        times.append(md.get_time() / units.fs) # time in fs
        
    md.attach(collect_data, interval=10)
    
    # Run for 300 steps (600 fs). 
    # NOTE: In a real paper, this is usually >50,000 steps! We use a small number for classroom time.
    md.run(300) 
    
    # Calculate MSD for Lithium only
    positions_np = np.array(positions)
    times_ps = np.array(times) / 1000.0
    
    pos_0_li = positions_np[0, li_indices, :]
    msd = []
    for t_step in range(len(positions_np)):
        pos_t_li = positions_np[t_step, li_indices, :]
        sq_dist = np.sum((pos_t_li - pos_0_li)**2, axis=1)
        msd.append(np.mean(sq_dist))
        
    # Extract D from slope of MSD vs time (slope = 6D)
    slope, intercept = np.polyfit(times_ps, msd, 1)
    D_A2_ps = slope / 6.0
    
    # Convert D from A^2/ps to cm^2/s
    D_cm2_s = D_A2_ps * 1e-4
    diffusion_coeffs.append(D_cm2_s)
    print(f"D({T}K) = {D_cm2_s:.2e} cm^2/s")

### 2.3 Arrhenius Plot & Extrapolation
Now we plot $\ln(D)$ vs $1000/T$ and extrapolate the line to $T = 300$ K.

In [ ]:
inv_T = 1000.0 / np.array(temperatures_K)
ln_D = np.log(diffusion_coeffs)

plt.figure(figsize=(6, 4))
plt.plot(inv_T, ln_D, 'ko', markersize=8, label='MD Data')

# Fit line to extract Activation Energy (Ea)
slope_arrh, intercept_arrh = np.polyfit(inv_T, ln_D, 1)
fit_line = slope_arrh * inv_T + intercept_arrh
plt.plot(inv_T, fit_line, 'r-', label='Arrhenius Fit')

plt.xlabel('1000 / T (K$^{-1}$)')
plt.ylabel('ln(D)')
plt.title('Arrhenius Plot for Li Diffusion')
plt.legend()
plt.grid(True)
plt.show()

# Extrapolate to 300 K
T_room = 300
inv_T_room = 1000.0 / T_room
ln_D_room = slope_arrh * inv_T_room + intercept_arrh
D_300K = np.exp(ln_D_room)

print(f"Extrapolated D at 300 K: {D_300K:.2e} cm^2/s")

### 2.4 Calculating Ionic Conductivity ($\sigma$)
The Nernst-Einstein equation relates $D$ and $\sigma$:
$$ \sigma = \frac{n q^2 D}{k_B T} $$

Where $n$ is the number density of Lithium ions ($N_{Li} / V$).

In [ ]:
# Number of Li atoms in our supercell
N_Li = len(li_indices)

# Volume of supercell in cm^3 (1 A^3 = 10^-24 cm^3)
vol_A3 = md_supercell.get_volume()
vol_cm3 = vol_A3 * 1e-24

# Number density of Li (ions / cm^3)
n = N_Li / vol_cm3

# Constants
q = const.e          # 1.602e-19 C
kb = const.k         # 1.38e-23 J/K

# Conductivity at 300 K
sigma_300K = (n * q**2 * D_300K) / (kb * T_room)

print(f"Estimated Ionic Conductivity at 300 K: {sigma_300K:.2e} S/cm")

# For real beta-Li3PS4, experimental values are typically around ~1e-4 S/cm.

### 🏋️‍♂️ Exercise 2: Compare with Another Solid Electrolyte
Now it's your turn. Compare LPS with another famous solid-state electrolyte: **LGPS** (Li$_{10}$GeP$_2$S$_{12}$).

1. Copy the code blocks from section 2.1 to 2.4.
2. Change the Materials Project ID to `mp-696128` (LGPS).
3. *Hint: LGPS unit cell is already quite large (50 atoms), so you don't need a large supercell. Use `solid_structure * (1, 1, 1)`.*
4. Re-run the MD loop, plot the Arrhenius equation, and calculate its ionic conductivity at room temperature.
5. Is LGPS better or worse than LPS based on your simulation?

In [ ]:
# Your code for Exercise 2 here:

